# Preflight evaluation
Runs on the stored dataset (`data/datasets/features_v1.parquet`). Every number here comes from the cells below; nothing is typed in by hand.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("lightgbm").setLevel(logging.ERROR)
from preflight_core.features.dataset import load_dataset
from preflight_core.models.evaluate import ablate_families, compare, per_repo, reliability_table
from preflight_core.models.zoo import MODELS

df = load_dataset()
df["label"].value_counts(dropna=False)

## Models vs baselines (rolling-origin CV)

In [ ]:
table, res = compare(
    {
        n: MODELS[n]
        for n in ["always_pass", "files_changed_gt_n", "logistic_regression", "lgbm_calibrated"]
    },
    df,
)
table

## Feature-family ablations

In [ ]:
ablate_families(MODELS["lgbm_calibrated"], df)

## Modelling decisions: SMOTE, monotone constraints, class weights

In [ ]:
compare(
    {
        n: MODELS[n]
        for n in [
            "lgbm_calibrated",
            "lgbm_smote",
            "lgbm_no_monotone",
            "lgbm_full_class_weight",
            "lgbm_no_class_weight",
        ]
    },
    df,
)[0]

## Reliability curve

In [ ]:
from IPython.display import Image

from preflight_core.models.plots import reliability_plot

reliability_plot(res["lgbm_calibrated"].oof, "reliability.png")
Image("reliability.png")

In [ ]:
reliability_table(res["lgbm_calibrated"].oof)

## Per repo

In [ ]:
per_repo(res["lgbm_calibrated"].oof)